# 04 - Gold Bookings Detail Refresh

Sanitized portfolio notebook for the Corporate Travel Booking Data Platform.

Before running this notebook in a real Databricks workspace, replace placeholder paths such as `<storage-account>` and confirm that the Unity Catalog schemas exist.
All outputs have been cleared before publishing.


In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col,
    lower,
    date_format,
    datediff,
    when
)

PROCESS_NAME = "gold_detail_refresh"
WATERMARK_TABLE = "corporate_travel_analytics.control.pipeline_watermark"

SILVER_CLEAN_TABLE = "corporate_travel_analytics.silver.bookings_clean"
SILVER_QUARANTINE_TABLE = "corporate_travel_analytics.silver.bookings_quarantine"

GOLD_ACTIVE_DETAIL_TABLE = "corporate_travel_analytics.gold.booking_cost_detail"
GOLD_CANCELLED_DETAIL_TABLE = "corporate_travel_analytics.gold.cancelled_booking_detail"

# =========================================================
# 1. READ LAST WATERMARK
# =========================================================
watermark_df = (
    spark.table(WATERMARK_TABLE)
    .filter(col("process_name") == PROCESS_NAME)
)

last_watermark = watermark_df.collect()[0]["last_ingest_time"]

print(f"Last watermark for {PROCESS_NAME}: {last_watermark}")

# =========================================================
# 2. READ CHANGED ROWS FROM SILVER CLEAN AND QUARANTINE
# =========================================================
changed_clean_df = (
    spark.table(SILVER_CLEAN_TABLE)
    .filter(col("ingest_time") > last_watermark)
)

changed_quarantine_df = (
    spark.table(SILVER_QUARANTINE_TABLE)
    .filter(col("ingest_time") > last_watermark)
)

has_clean_changes = changed_clean_df.limit(1).count() > 0
has_quarantine_changes = changed_quarantine_df.limit(1).count() > 0

if not has_clean_changes and not has_quarantine_changes:
    print("No Gold detail changes found. Nothing to process.")
else:
    if has_clean_changes:
        print(f"Changed clean rows found: {changed_clean_df.count()}")
    if has_quarantine_changes:
        print(f"Changed quarantine rows found: {changed_quarantine_df.count()}")

    # =========================================================
    # 3. SPLIT CLEAN ROWS INTO ACTIVE AND CANCELLED
    # =========================================================
    active_clean_changed_df = changed_clean_df.filter(
        lower(col("booking_status")) != "cancelled"
    )

    cancelled_clean_changed_df = changed_clean_df.filter(
        lower(col("booking_status")) == "cancelled"
    )

    # =========================================================
    # 4. TRANSFORM ACTIVE CLEAN ROWS TO GOLD ACTIVE DETAIL FORMAT
    # =========================================================
    active_detail_df = (
        active_clean_changed_df
        .select(
            col("booking_id"),
            col("employee_id"),
            col("employee_name"),
            col("department"),
            col("employee_email"),
            col("trip_type"),
            col("origin"),
            col("destination"),
            col("departure_ts"),
            col("return_ts"),
            col("travel_mode"),
            col("airline"),
            col("hotel_name"),
            col("hotel_city"),
            col("check_in"),
            col("check_out"),
            col("currency"),
            col("flight_cost"),
            col("hotel_cost"),
            col("total_cost"),
            col("approval_status"),
            col("approved_by"),
            col("approval_ts"),
            col("booking_status"),
            col("booking_created_ts"),
            col("poc_email"),
            date_format(col("booking_created_ts"), "yyyy-MM").alias("booking_month"),
            datediff(col("check_out"), col("check_in")).alias("hotel_nights"),
            when(col("total_cost") >= 100000, "High Cost")
                .when(col("total_cost") >= 50000, "Medium Cost")
                .otherwise("Low Cost")
                .alias("cost_band")
        )
    )

    # =========================================================
    # 5. TRANSFORM CANCELLED CLEAN ROWS TO GOLD CANCELLED DETAIL FORMAT
    # =========================================================
    cancelled_detail_df = (
        cancelled_clean_changed_df
        .select(
            col("booking_id"),
            col("employee_id"),
            col("employee_name"),
            col("department"),
            col("employee_email"),
            col("trip_type"),
            col("origin"),
            col("destination"),
            col("departure_ts"),
            col("return_ts"),
            col("travel_mode"),
            col("airline"),
            col("hotel_name"),
            col("hotel_city"),
            col("check_in"),
            col("check_out"),
            col("currency"),
            col("flight_cost"),
            col("hotel_cost"),
            col("total_cost"),
            col("approval_status"),
            col("approved_by"),
            col("approval_ts"),
            col("booking_status"),
            col("booking_created_ts"),
            col("poc_email"),
            date_format(col("booking_created_ts"), "yyyy-MM").alias("booking_month"),
            datediff(col("check_out"), col("check_in")).alias("hotel_nights"),
            when(col("total_cost") >= 100000, "High Cost")
                .when(col("total_cost") >= 50000, "Medium Cost")
                .otherwise("Low Cost")
                .alias("cost_band")
        )
    )

    # =========================================================
    # 6. CREATE GOLD DETAIL TABLES IF NOT EXISTS
    # =========================================================
    if not spark.catalog.tableExists(GOLD_ACTIVE_DETAIL_TABLE):
        active_detail_df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(GOLD_ACTIVE_DETAIL_TABLE)
        print(f"Created target table: {GOLD_ACTIVE_DETAIL_TABLE}")

    if not spark.catalog.tableExists(GOLD_CANCELLED_DETAIL_TABLE):
        cancelled_detail_df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(GOLD_CANCELLED_DETAIL_TABLE)
        print(f"Created target table: {GOLD_CANCELLED_DETAIL_TABLE}")

    # =========================================================
    # 7. MERGE ACTIVE BOOKINGS INTO GOLD ACTIVE DETAIL
    # =========================================================
    if active_detail_df.limit(1).count() > 0:
        active_target = DeltaTable.forName(spark, GOLD_ACTIVE_DETAIL_TABLE)

        active_target.alias("t").merge(
            active_detail_df.alias("s"),
            "t.booking_id = s.booking_id"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()

        print("Merged active rows into gold.booking_cost_detail")

        # Remove same booking_id from cancelled detail if now active
        active_ids_df = active_detail_df.select("booking_id").distinct()
        active_ids_df.createOrReplaceTempView("active_ids_to_remove_from_cancelled")

        spark.sql(f"""
            DELETE FROM {GOLD_CANCELLED_DETAIL_TABLE}
            WHERE booking_id IN (
                SELECT booking_id FROM active_ids_to_remove_from_cancelled
            )
        """)

        print("Removed active booking_ids from gold.cancelled_booking_detail")

    # =========================================================
    # 8. MERGE CANCELLED BOOKINGS INTO GOLD CANCELLED DETAIL
    # =========================================================
    if cancelled_detail_df.limit(1).count() > 0:
        cancelled_target = DeltaTable.forName(spark, GOLD_CANCELLED_DETAIL_TABLE)

        cancelled_target.alias("t").merge(
            cancelled_detail_df.alias("s"),
            "t.booking_id = s.booking_id"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()

        print("Merged cancelled rows into gold.cancelled_booking_detail")

        # Remove same booking_id from active detail if now cancelled
        cancelled_ids_df = cancelled_detail_df.select("booking_id").distinct()
        cancelled_ids_df.createOrReplaceTempView("cancelled_ids_to_remove_from_active")

        spark.sql(f"""
            DELETE FROM {GOLD_ACTIVE_DETAIL_TABLE}
            WHERE booking_id IN (
                SELECT booking_id FROM cancelled_ids_to_remove_from_active
            )
        """)

        print("Removed cancelled booking_ids from gold.booking_cost_detail")

    # =========================================================
    # 9. REMOVE INVALID BOOKINGS FROM BOTH GOLD DETAIL TABLES
    # =========================================================
    if has_quarantine_changes:
        invalid_ids_df = changed_quarantine_df.select("booking_id").distinct()
        invalid_ids_df.createOrReplaceTempView("invalid_ids_for_gold_delete")

        spark.sql(f"""
            DELETE FROM {GOLD_ACTIVE_DETAIL_TABLE}
            WHERE booking_id IN (
                SELECT booking_id FROM invalid_ids_for_gold_delete
            )
        """)

        spark.sql(f"""
            DELETE FROM {GOLD_CANCELLED_DETAIL_TABLE}
            WHERE booking_id IN (
                SELECT booking_id FROM invalid_ids_for_gold_delete
            )
        """)

        print("Removed invalid booking_ids from both Gold detail tables")

    # =========================================================
    # 10. UPDATE WATERMARK
    # =========================================================
    ingest_frames = []

    if has_clean_changes:
        ingest_frames.append(changed_clean_df.select("ingest_time"))

    if has_quarantine_changes:
        ingest_frames.append(changed_quarantine_df.select("ingest_time"))

    combined_ingest_df = ingest_frames[0]
    for extra_df in ingest_frames[1:]:
        combined_ingest_df = combined_ingest_df.unionByName(extra_df)

    max_ingest_time = combined_ingest_df.agg({"ingest_time": "max"}).collect()[0][0]

    spark.sql(f"""
        DELETE FROM {WATERMARK_TABLE}
        WHERE process_name = '{PROCESS_NAME}'
    """)

    spark.sql(f"""
        INSERT INTO {WATERMARK_TABLE}
        VALUES ('{PROCESS_NAME}', TIMESTAMP('{max_ingest_time}'))
    """)

    print(f"Updated watermark for {PROCESS_NAME} to {max_ingest_time}")
